
---

## **Part 1 — Exploring Different Modalities / Representations of Network Traffic**

Synthetic network traffic generation is useful for many applications such as dataset augmentation, network testing, and resource management. Many existing generation methods treat traffic generation as either a time-series prediction task or an autoregressive modeling task. In these approaches, models are trained directly on structured representations of packets—one common example is **nPrint**, a tabular format that encodes packet header fields from PCAP traces into machine-learning-friendly numerical vectors.

In an **nPrint**, each row represents one packet in a trace. The entire nPrint file (a large CSV) represents all packets of that trace in sequential order. Header fields are encoded using one-hot–like binary indicators (e.g., `1`, `0`, or `01`), making them easy to feed into ML models.

However, general-purpose ML models—even advanced time-series architectures and transformers—often struggle to capture the *complex dependencies* present in real network traffic:

* **Local dependencies**: relationships among columns within a single row (i.e., dependencies among header fields of a single packet).
* **Global dependencies**: relationships across rows (i.e., the evolution of packets within the same flow).
  For example: if the first packet of a flow uses TCP, the second packet in that same flow should also be TCP; sequence numbers, flags, and flow identifiers evolve in structured ways over time.

While sequential ML models struggle with these multi-scale dependencies, **vision models** (e.g., diffusion models) excel at capturing both local and global structure when data is presented spatially—like an image. By converting nPrint traces into 2D PNG images, we can take advantage of the strong representational capabilities of image models and generate synthetic traffic using visual generative approaches.

---

### **Your Task for Part 1**

In this part of the assignment, you will:

1. **Inspect the raw nPrint files** (found in the `real_nprints` directory).
2. **Understand how each CSV row and column corresponds to packet-level metadata.**
3. **Follow the provided conversion pipeline** that transforms these nPrint CSVs into PNG image representations suitable for use with diffusion models and other visual architectures.
4. **Take an already generated set of image-representation of images and convert them back into nprint representation for downstream task utilization**

This will help you understand why converting network traces to images can unlock generative modeling capabilities that traditional ML approaches struggle with.

Q1:
First, download and unzip the data you will need from (https://drive.google.com/file/d/1hY6nNXEYOwl1l-O_nCknO9xezcHr6ZXi/view?usp=sharing)
In your own words, describe how an nPrint CSV encodes a network trace.
Why might a 2D image representation capture structural relationships that a row-by-row CSV cannot?

I'm not sure if it actually is a way to capture structural relationships that you CAN'T capture with a CSV, but the representation definitely makes it a lot easier to read on the whole, with all of the indicators encoded as a certain color in the file. It's easy to see, depending on what colors you have encoded to each threshold, a larger overview of the image is readily available without having to scroll through 10s of thousands of packet values.

Q2. Design a method for converting nPrint representations of traces (in the folder real_nprints) into image representations.
Your image representation should use only the first 1024 packets from each trace to avoid producing images that are too large.
Save the images into a folder called './nd_data/student_converted_images'

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image

# ---------------------------------------------------------------
# My approach: render each nPrint as a color image, where each row
# is a packet and each column is a header bit/field. I follow the
# same three-color palette that color_processor.py snaps pixels
# toward (pure red, green, and blue), mapping the three possible
# nPrint values directly onto those colors:
#   1  (bit set)      -> red   (255, 0, 0, 255)
#   0  (bit unset)     -> green (0, 255, 0, 255)
#  -1  (missing/N/A)   -> blue  (0, 0, 255, 255)
# This keeps a simple, information-preserving 1-to-1 mapping
# between nPrint values and pixel colors.
# ---------------------------------------------------------------

NUM_PACKETS = 1024

VALUE_TO_COLOR = {
    1: (255, 0, 0, 255),   # red
    0: (0, 255, 0, 255),   # green
    -1: (0, 0, 255, 255),  # blue
}


def value_to_color(x):
    try:
        return VALUE_TO_COLOR[int(x)]
    except (ValueError, KeyError):
        return VALUE_TO_COLOR[-1]  # treat anything unexpected as "missing"


def nprint_to_image_array(df):
    # Drop any identifying IP address columns, if present, so the
    # image represents header *structure* rather than raw addresses.
    id_substrings = ["ipv4_src", "ipv4_dst", "ipv6_src", "ipv6_dst", "src_ip", "dst_ip"]
    df = df.drop(columns=[c for c in df.columns if any(s in c for s in id_substrings)])

    # Use only the first NUM_PACKETS packets (rows) of the trace.
    df = df.iloc[:NUM_PACKETS, :]

    height, width = df.shape
    arr = np.array(df.map(value_to_color).to_numpy().tolist(), dtype=np.uint8)
    arr = arr.reshape(height, width, 4)

    # Pad shorter traces with "missing packet" rows so every image
    # has the same height, without inflating traces that are already long.
    if arr.shape[0] < NUM_PACKETS:
        pad = np.full((NUM_PACKETS - arr.shape[0], width, 4), VALUE_TO_COLOR[-1], dtype=np.uint8)
        arr = np.vstack([arr, pad])

    return arr


def convert_nprint_dir_to_images(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for file_path in sorted(glob.glob(os.path.join(input_dir, "*.nprint"))):
        file_name = os.path.basename(file_path)
        print(f"Processing {file_name}")
        try:
            df = pd.read_csv(file_path, index_col=0)
            if df.empty:
                continue

            arr = nprint_to_image_array(df)
            img = Image.fromarray(arr, mode="RGBA")

            output_file = os.path.join(output_dir, file_name.replace(".nprint", ".png"))
            img.save(output_file)
        except Exception as e:
            print(f"Failed to process {file_name}: {e}")
            continue


convert_nprint_dir_to_images("./nd_data/real_nprints", "./nd_data/student_converted_images")


In [1]:
# The following is a pre-defined script used in NetDiffusion that will convert all of the provided real nprints into image representations. Run this code and observe the output
!python ./scripts/nprint_to_png.py -i ./nd_data/real_nprints/ -o ./nd_data/real_traffic_images

Processing teams_1024_10.nprint
/Users/domi/Desktop/CMSC25422/CMSC-25422/netdiffusion 2/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing twitch_1024_7.nprint
/Users/domi/Desktop/CMSC25422/CMSC-25422/netdiffusion 2/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing netflix_1024_15.nprint
/Users/domi/Desktop/CMSC25422/CMSC-25422/netdiffusion 2/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing twitter_1024_17.nprint
/Users/domi/Desktop/CMSC25422/CMSC-25422/netdiffusion 2/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_d

Q3: Now that you have seen how NetDiffusion converts nPrints into images, compare their method with the approach you designed in Q2. What are the advantages and disadvantages of each, especially in terms of what might help or hinder a vision model’s ability to learn?

The largest difference is that NetDiffusions model, when receiving inputs that aren't 1,0, or -1, will encode the value as the fourth value of the rgb code. The model Claude devised simply takes any other value than 1 or 0 and encodes it as missing. I think this difference can change a lot when you aren't looking at it with the naked eye, because there is stuff that the vision model can see that a person can't. I can't see that fourth value, so I think that they look the same. However the vision model should still be able to see that, which is why I believe some of the generated traffic images have 4 colors: red, green, blue, and purple. It's using the extra information to kind of create a fourth threshold. I'm not sure if this would help it learn, but I wonder if there being more information would make it easier to decode the images into a more accurate description of the original data.


---

## **Part 2 — Converting Generated Images Back Into Usable Format**

In the first part of the assignment, you explored how network traces in nPrint format can be transformed into image representations suitable for vision-based generative models. In Part 2, we focus on the reverse process: taking synthetic images produced by these models and converting them back into structured network representations.

This step is crucial because real-world applications do not operate on images—they require valid, interpretable packet traces that can be analyzed, replayed, or integrated into downstream tools.

---

### **Your Task for Part 2**

In this part of the assignment, you will:

1. **Convert generated images back into the original nPrint representation.**
   You will follow a scripted pipeline that translates pixel intensities and color channels back into binary header fields, reconstructing the packet-level structure of the trace.

2. **Apply essential post-processing techniques to correct errors introduced by diffusion models.**
   Generated images are rarely perfect—vision models may introduce color drift, pixel misalignment, noise, or structural artifacts.
   You will observe how heuristic correction, formatting enforcement, and reconstruction steps ensure that the converted nPrints become:

   * syntactically valid,
   * structurally consistent,
   * and replayable.

Across this section, your goal is to understand **why the reverse transformation is fragile**, which types of artifacts break reversibility, and how post-processing logic helps repair or compensate for generative errors.

For simplicity of this assignment, we have trainined and generated the images for you. If you have sufficient GPU access and want to try fine-tuning the model and generating the images yourself, feel free to take a look at the public repo (https://github.com/noise-lab/NetDiffusion).


Q4: We have taken the images converted by NetDiffusion and trained a LoRA-fine-tuned Stable Diffusion model (with ControlNet) to generate synthetic traffic images for you. These generated samples are stored in generated_traffic_images/.
Compare these generated images visually with the real images you saw earlier in real_traffic_images/.

Do you notice anything different between the real and generated traffic images? What immediately stands out as potentially problematic if we attempt to convert these generated images back into nPrint format? (Descriptive Only)

There are two main issues that I can see:  

The first is the appearance of a second color. With the manual encoding, we are able to go through, pixel by pixel, and figure whether each is inside the range we want. With the generated image, a fourth color makes it so that we probably can't use the same function we would use to decode the manual images, because there's an appearance of something that is simply not there when we have to edit the function to be able to take more things as an input.

The second is the smoothing present in the generated images. In the manual images, it's very clear that the pixels are all individually colored. In the generated images, some of the colors melt into each other which seems like it would be a dangerous for trying to extract explicit data out of them.

Q5: If you were to design a method to convert generated images back into nPrints, how would you do it? Explain your approach and describe how your method addresses the concerns you raised in Q4. (Descriptive only)

If I were to make a method to turn the generated images back into nPrints, I would take an example original nPrint, 1024 rows x however many columns we are expecting, depending on the # of headers, and use that to design the way my function will navigate the image. Once I've devised that, I would divide generated image into that many sections (essentially moving thru the image pixel by pixel as if it were a manual nPrint) By doing this, I think I could minimize the issue of the extra color by just taking it as a missing value, which is essentially what the function I made using Claude did. To solve the smoothing issue, I would see if I can find a way through some library (I think you can just do this by making the image an array) and matching the "nearest" color to the value. For example, "true" red is (255,0,0), so if I stumble upon a pixel that's color code is like, (220, 34, 102), I would just assume it was meant to be a red pixel. The simplest way to do this is just by taking the highest value, but if there was something with equal values I would see if I could figure out what pakcet header we're currently in, and see what a reasonable value for it would be based on the surrounding values. The only issue with that is how you decide what a "reasonable" value is, and if I had time and it was an important project, I would probably use an ML agent to scan my manually converted, real traffic and compare the pixel locations to what the most common one is across manually generated images that match. This seems kind-of deep but I think it would be the most interesting way to keep it "accurate."

Below are a set of pre-written scripts that perform the necessary post-generation augmentation and processing on the synthetic images you obtained from the diffusion model. These scripts handle tasks such as color normalization/augmentation and conversion from generated images back into nPrint format.
(The PCAP step is optional — you may run it if you are interested in observing or replaying the reconstructed traffic.)

In [2]:
# Step 1: Color Augmentation
!python ./scripts/color_processor.py \
  --input_dir="./nd_data/generated_traffic_images" \
  --output_dir="./nd_data/color_corrected_generated_traffic_images"

Processed 100 images.


In [3]:
# Step 2: Image-to-nPrint Conversion
!python ./scripts/image_to_nprint.py \
  --org_nprint ./scripts/column_example.nprint \
  --input_dir ./nd_data/color_corrected_generated_traffic_images \
  --output_dir ./nd_data/generated_nprint

Processing ./nd_data/color_corrected_generated_traffic_images/teams_5.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/teams_5.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/twitch_3.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/twitch_3.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/zoom_7.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/zoom_7.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/zoom_6.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/zoom_6.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/twitch_2.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/twitch_2.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/teams_4.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/teams_4.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/teams_6.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/teams_6

Q6: You may now read through the provided scripts in color_processor.py (color augmentation / normalization) and image_to_nprint.py (image → nPrint reconstruction).
How do the post-processing methods implemented in these scripts compare to the approach you proposed in Q5?
Describe the pros and cons of both methods and highlight any differences in design philosophy, robustness, or assumptions.

The color processor function did basically exactly what I was suggesting, philosophy wise, which was lowkey pretty gratifying. The png_to_nprint was also pretty similar, except without the extra ML application. Since it assumes that extra, non-missing value is encoded, it allows the function to decode both manual and synthetically generated pngs the same way. I'm curious if missing that "A" value causes me to miss out on any extra data that could be encoded into the image without necessarily changing the image. As far as I can tell, though, it's a pretty front-to-back flip of the nPrint_to_png function, though. It seems simple-ish to implement as long as you are able to come up with the idea of the nprint_to_png workflow, as you just reverse the order and function of the steps.


---

# **Part 3 — Using Real and Synthetic nPrints for Application Classification**

In the previous parts, you learned how network traces can be converted between nPrint and image representations, generated using diffusion models, and reconstructed back into nPrint format.
Now, you will evaluate how useful these generated nPrints are for downstream **machine learning tasks**.

Each nPrint file—whether real or generated—is labeled with the **application** that produced the traffic (e.g., `amazon_1.nprint` means this sample came from Amazon traffic).
In this section, you will treat each **entire nPrint file as a single sample** and build a simple ML pipeline to classify application labels.

To simplify the task, you will restrict your model to use **only the first 3 packets** (3 rows) from each nPrint.
This mimics “early packet classification,” where only the beginning of a flow is available.

---


Q7:

You now have access to both `real_nprints/` and `generated_nprint/`.
Notice that in both directories, files are labeled using the application associated with that nPrint (e.g., `amazon_1.nprint`).
Treat each **nPrint file** as one sample.

**Design an ML pipeline that trains a model using *synthetic nPrints* (from `generated_nprint/`) and evaluates its performance on *real nPrints* (from `real_nprints/`) to predict the correct application label.**

Your pipeline should:

1. Use only the **first 3 packets (first 3 rows)** of each nPrint file as input features.
2. Train a classifier on real data.
3. Test the classifier on generated data.
4. Report how well the classifier performs.

We have already written the script to load the real and generated nprints into DataFrame for you.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# Safe conversion for nPrint cell
# ---------------------------------------------------------------
def safe_convert(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()

    if x in ["0", "1", "-1"]:
        return int(x)

    if all(c in "01" for c in x) and len(x) <= 16:
        return int(x, 2)

    if x.lstrip("-").isdigit():
        return int(x)

    return 0


# ---------------------------------------------------------------
# Get original column names from a reference nPrint
# ---------------------------------------------------------------
def get_original_columns(example_path="nd_data/real_nprints"):
    first_file = glob.glob(os.path.join(example_path, "*.nprint"))[0]

    df = pd.read_csv(first_file)

    # Drop index column if present (like "Unnamed: 0")
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    return list(df.columns)


# ---------------------------------------------------------------
# Load nPrint → first 3 rows → flatten with prefixed column names
# ---------------------------------------------------------------
def load_nprint_with_colnames(path, base_cols, num_rows=3):
    df = pd.read_csv(path, dtype=str, low_memory=False)

    # Drop "Unnamed: 0" if present
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    df = df.iloc[:num_rows, :]            # first 3 packets  
    df = df.map(safe_convert)             # clean convert  

    # Build prefixed column names
    pkt_cols = []
    for pkt in range(1, num_rows + 1):
        pkt_cols.extend([f"pkt{pkt}_{c}" for c in base_cols])

    # Flatten 3×columns into 1 vector
    flat = df.values.flatten()

    return flat, pkt_cols


# ---------------------------------------------------------------
# Load entire directory into a DataFrame (with labels)
# ---------------------------------------------------------------
def load_directory_as_df(directory, base_cols):
    rows = []
    labels = []
    colnames_set = None

    for path in glob.glob(os.path.join(directory, "*.nprint")):
        label = os.path.basename(path).split("_")[0]

        flat, cn = load_nprint_with_colnames(path, base_cols)
        rows.append(flat)
        labels.append(label)

        if colnames_set is None:
            colnames_set = cn   # only set once

    df = pd.DataFrame(rows, columns=colnames_set)
    df["label"] = labels
    return df


# ---------------------------------------------------------------
# FINAL: Load real + synthetic DataFrames
# ---------------------------------------------------------------
base_cols = get_original_columns("nd_data/real_nprints")

df_synth = load_directory_as_df("nd_data/generated_nprint", base_cols)
df_real  = load_directory_as_df("nd_data/real_nprints", base_cols)

print("Synthetic DF:", df_synth.shape)
print(df_synth.head())

print("\nReal DF:", df_real.shape)
print(df_real.head())


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


feature_cols = [c for c in df_synth.columns if c != "label"] #reindex to guranteed the same columns in both sets of data

X_train = df_synth.reindex(columns=feature_cols, fill_value=0)
y_train = df_synth["label"]
#turning synthetic data into the training data (already sectioned for us))


X_test = df_real.reindex(columns=feature_cols, fill_value=0)
y_test = df_real["label"]
#turning real data into the testing data

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
#make a random forest classifier / load training data

y_pred = clf.predict(X_test)

print(f"Accuracy (train=synthetic, test=real): {accuracy_score(y_test, y_pred):.3f}")
print()
print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels_sorted = sorted(y_test.unique())
print("Confusion matrix (rows = true label, columns = predicted label):")
print("Labels:", labels_sorted)
print(confusion_matrix(y_test, y_pred, labels=labels_sorted))


Accuracy (train=synthetic, test=real): 0.290

Classification report:
              precision    recall  f1-score   support

      amazon       0.00      0.00      0.00        20
    facebook       0.55      0.30      0.39        20
   instagram       0.50      0.25      0.33        20
        meet       0.75      0.30      0.43        20
     netflix       0.50      0.10      0.17        20
       teams       0.27      0.95      0.42        20
      twitch       0.00      0.00      0.00        20
     twitter       0.22      1.00      0.35        20
     youtube       0.00      0.00      0.00        20
        zoom       0.00      0.00      0.00        20

    accuracy                           0.29       200
   macro avg       0.28      0.29      0.21       200
weighted avg       0.28      0.29      0.21       200

Confusion matrix (rows = true label, columns = predicted label):
Labels: ['amazon', 'facebook', 'instagram', 'meet', 'netflix', 'teams', 'twitch', 'twitter', 'youtube', 'zo

I'm curious as to why there was so much incorrectly labeled twitter data. Also, I wonder if allowing us to use the entire traces would increase accuracy significantly, or if the issue was just the fact that it was trained on the synthesized data. Either way, I think the results are definitely better than I would have expected with seeing the original, synthesized images compared to the manually created images from the real traffic. I also wonder what would happen if we made synthesized nPrint representations, and then manually converted them to images. But also, that seems kind of pointless since we are making them back into nPrints anyways. I wonder if a modern AI image generation would be able to create better versions of these pngs, since I think that technology advances an incredible degree every year or so, and I think we have also gotten a lot better at our ability to prompt AI models as well. Overall, I think this was a really interesting look into a way to create data that I don't think I would have considered at all, nPrint -> png -> training -> synthesization -> nPrint = a (seemingly) faster, easier to store, and easier to overview, way to look at and create more data to solve the issue of not having enough (or really, not being able to source enough data) to feed to our models while still respecting user privacy on the networks we might be overseeing.